## Imports

In [1]:
import argparse
import os
import pathlib
import sys

import imageio
import napari

# import matplotlib.pyplot as plt
import numpy as np
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.file_reading import *
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from moviepy import VideoFileClip
from napari_animation import Animation
from napari_animation.easing import Easing
from PIL import Image

root_dir, in_notebook = init_notebook()

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    input_subparent_name = args["input_subparent_name"]
    mask_subparent_name = args["mask_subparent_name"]
    amimation_subparent_name = args["amimation_subparent_name"]
    check_for_missing_args(
        well_fov=well_fov,
        patient=patient,
        input_subparent_name=input_subparent_name,
        mask_subparent_name=mask_subparent_name,
        amimation_subparent_name=amimation_subparent_name,
    )

else:
    print("Running in a notebook")
    well_fov = "C4-2"
    patient = "NF0014_T1"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    amimation_subparent_name = "animations"

image_dir = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
).resolve(strict=True)
label_dir = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
).resolve(strict=True)
mp4_file_dir = pathlib.Path(
    f"{root_dir}/data/{patient}/{amimation_subparent_name}/mp4/{well_fov}/"
).resolve()
gif_file_dir = pathlib.Path(
    f"{root_dir}/data/{patient}/{amimation_subparent_name}/gif/{well_fov}/"
).resolve()

mp4_file_dir.mkdir(parents=True, exist_ok=True)
gif_file_dir.mkdir(parents=True, exist_ok=True)
tmp_output_path = "output.zarr"

Running in a notebook


In [3]:
def mp4_to_gif(input_mp4: pathlib.Path, output_gif: pathlib.Path, fps: int = 30):
    """
    Convert an mp4 file to a gif file using moviepy.
    Parameters
    ----------
    input_mp4 : pathlib.Path
        The path to the input mp4 file.
    output_gif : pathlib.Path
        The path to the output gif file.
    fps : int, optional
        The frames per second for the output gif file, by default 30.

    Returns
    -------
    None
    """
    with VideoFileClip(str(input_mp4)) as clip:
        width, height = clip.size
        side = min(width, height)

        # keep codec-friendly dimensions
        if side % 2 != 0:
            side -= 1

        x1 = int((width - side) // 2)
        y1 = int((height - side) // 2)

        square_clip = clip.cropped(
            x1=x1,
            y1=y1,
            x2=x1 + side,
            y2=y1 + side,
        )

        # Write only GIF (do not overwrite source mp4)
        square_clip.write_gif(
            str(output_gif),
            fps=fps,
            loop=0,
        )

In [4]:
def animate_view(
    viewer: napari.Viewer,
    output_path_name: str,
    steps: int = 30,
    easing: str = "linear",
    dim: int = 3,
):
    """
    Animate a napari viewer by rotating around the y-axis and then back to the original position.
    Parameters
    ----------
    viewer : napari.Viewer
        The napari viewer to animate.
    output_path_name : str
        The path to save the output mp4 file.
    steps : int, optional
        The number of steps for each keyframe, by default 30.
    easing : str, optional
        The easing style for the animation, by default "linear".
    dim : int, optional
        The number of dimensions to display, by default 3.
    Returns
    -------
    None
    """
    animation = Animation(viewer)
    if easing == "linear":
        ease_style = Easing.LINEAR
    else:
        raise ValueError(f"Invalid easing style: {easing}")

    viewer.dims.ndisplay = dim
    # rotate around the y-axis
    viewer.camera.angles = (0.0, 0.0, 90.0)  # (z, y, x) axis of rotation
    animation.capture_keyframe(steps=steps, ease=ease_style)

    viewer.camera.angles = (0.0, 180.0, 90.0)
    animation.capture_keyframe(steps=steps, ease=ease_style)

    viewer.camera.angles = (0.0, 360.0, 90.0)
    animation.capture_keyframe(steps=steps, ease=ease_style)

    viewer.camera.angles = (0.0, 0.0, 270.0)
    animation.capture_keyframe(steps=steps, ease=ease_style)

    viewer.camera.angles = (0.0, 0.0, 90.0)
    animation.capture_keyframe(steps=steps, ease=ease_style)

    animation.animate(output_path_name, canvas_only=True)

In [5]:
label_dir
output_path = "output.zarr"
channel_map = {
    "405": "Nuclei",
    "488": "Endoplasmic Reticulum",
    "555": "Actin, Golgi, and plasma membrane (AGP)",
    "640": "Mitochondria",
    "TRANS": "Brightfield",
}
scaling_values = [1, 0.1, 0.1]
image_metadata = f"{patient}_{well_fov}"

In [6]:
image_return_dict = read_in_channels(
    find_files_available(image_dir),
    channel_dict={
        "DNA": "405",
        "Endoplasmic_Reticulum": "488",
        "AGP": "555",
        "Mitochondria": "640",
    },
    channels_to_read=[
        "DNA",
        "Endoplasmic_Reticulum",
        "AGP",
        "Mitochondria",
    ],
)

mask_return_dict = read_in_channels(
    find_files_available(label_dir),
    channel_dict={
        "Nuclei_mask": "nuclei",
        "Cell_mask": "cell",
        "Cytoplasm_mask": "cytoplasm",
        "Organoid_mask": "organoid",
    },
    channels_to_read=[
        "Nuclei_mask",
        "Cell_mask",
        "Cytoplasm_mask",
        "Organoid_mask",
    ],
)

headless = False
viewer = napari.Viewer(ndisplay=3, show=bool(not headless))

/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/skimage/util/dtype.py:529: UserWarning: Downcasting int64 to uint16 without scaling because max value 1 fits in uint16
  return _convert(image, np.uint16, force_copy)


In [7]:
for image_name, image_array in image_return_dict.items():
    viewer.add_image(
        image_array,
        name=f"{image_metadata}_{image_name}",
        scale=scaling_values,
    )
for mask_name, mask_array in mask_return_dict.items():
    viewer.add_labels(
        mask_array,
        name=f"{image_metadata}_{mask_name}",
        scale=scaling_values,
    )

In [8]:
# make the viewer full screen
viewer.window._qt_window.showMaximized()
# hide the layer controls
viewer.window._qt_viewer.dockLayerList.setVisible(False)
# hide the layer controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)

# set the viewer to a set window size
viewer.window._qt_window.resize(1000, 1000)
viewer.camera.zoom = 10.0

In [9]:
# get the layer names in the viewer
layer_names = [layer.name for layer in viewer.layers]
# set all layers to not visible
for layer_name in layer_names:
    print(f"Setting {layer_name} to not visible")
    viewer.layers[layer_name].visible = False

Setting NF0014_T1_C4-2_DNA to not visible
Setting NF0014_T1_C4-2_Endoplasmic_Reticulum to not visible
Setting NF0014_T1_C4-2_AGP to not visible
Setting NF0014_T1_C4-2_Mitochondria to not visible
Setting NF0014_T1_C4-2_Nuclei_mask to not visible
Setting NF0014_T1_C4-2_Cell_mask to not visible
Setting NF0014_T1_C4-2_Cytoplasm_mask to not visible
Setting NF0014_T1_C4-2_Organoid_mask to not visible


In [10]:
for layer_name in layer_names:
    viewer.layers[layer_name].visible = True
    # change the brightness and contrast for the raw signal layers
    if ".tif" in layer_name:
        save_name = layer_name.split(".tif")[0]
    else:
        save_name = layer_name

    # map the layer name to the channel name
    if "DNA" in layer_name:
        save_name = "DNA"
    elif "Endoplasmic" in layer_name:
        save_name = "ER"
    elif "AGP" in layer_name:
        save_name = "AGP"
    elif "Mitochondria" in layer_name:
        save_name = "mitochondria"
    else:
        save_name = layer_name

    save_path = pathlib.Path(f"{mp4_file_dir}/{well_fov}_{save_name}_animation.mp4")
    if "Mito" in layer_name:
        # increase contrast for the mitochondria
        viewer.layers[layer_name].contrast_limits = (0, 20000)
    animate_view(viewer, save_path, steps=30, easing="linear")
    viewer.layers[layer_name].visible = False
print("All layers animated")

Rendering frames...


 12%|█▏        | 15/121 [00:01<00:07, 14.41it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
 36%|███▋      | 44/121 [00:02<00:05, 14.32it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
100%|██████████| 121/121 [00:08<00:00, 14.65it/s]


Rendering frames...


 12%|█▏        | 14/121 [00:00<00:06, 15.99it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
 36%|███▋      | 44/121 [00:02<00:04, 16.85it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
100%|██████████| 121/121 [00:07<00:00, 15.78it/s]


Rendering frames...


...

Rendering frames...


 12%|█▏        | 15/121 [00:00<00:06, 16.96it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
 37%|███▋      | 45/121 [00:02<00:03, 19.34it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
100%|██████████| 121/121 [00:06<00:00, 17.68it/s]


Rendering frames...


 12%|█▏        | 15/121 [00:00<00:05, 19.28it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
 37%|███▋      | 45/121 [00:02<00:03, 19.15it/s]/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.12/site-packages/napari_animation/interpolation/base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
100%|██████████| 121/121 [00:06<00:00, 18.29it/s]


All layers animated


In [11]:
# get all gifs in the directory
mp4_file_path = list(pathlib.Path(mp4_file_dir).rglob("*.mp4"))
for mp4_file in mp4_file_path:
    # change the path to the gif directory
    mp4_file = pathlib.Path(mp4_file)
    gif_file = pathlib.Path(gif_file_dir / f"{mp4_file.stem}.gif")
    mp4_file = str(mp4_file)
    gif_file = str(gif_file)
    mp4_to_gif(mp4_file, gif_file)

MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_ER_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_mitochondria_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_NF0014_T1_C4-2_Cell_mask_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_NF0014_T1_C4-2_Cytoplasm_mask_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_DNA_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_NF0014_T1_C4-2_Organoid_mask_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_NF0014_T1_C4-2_Nuclei_mask_animation.gif with imageio.


MoviePy - Building file /home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/animations/gif/C4-2/C4-2_AGP_animation.gif with imageio.
